# 06 — Severe dry-spell risk: mean residual duration maps

Produces `figures/mean_residual_duration_long_dry_spells.pdf` (Fig. 14). For each station, we compute $\mathbb{E}[\tau^{(0)} - d \mid \tau^{(0)} > d]$ at thresholds $d \in \{20, 40, 60\}$ under:
- the two-state first-order Markov baseline (closed form, $1/\hat p_{\mathrm{geom}}$);
- the hdeGPD model (bounds from Appendix I, tightened to $10^{-5}$).

Large background circle = geometric; small foreground circle = hdeGPD. Colour-scale blue (short) → red (long).

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve().parents[1]))
import json
import numpy as np
import pandas as pd
from article_code.util_files import config, plotting
from article_code.util_files.spell_models import *
from article_code.util_files.statistics import *
NAME_STATION_EXAMPLE = config.STATION_EXAMPLE

# Load saved results from notebook 01.
name_folder_save_results = (f"fit_south_europe_subset_excess_over_{config.WET_DAY_THRESHOLD}")
path_folder_save_results = config.RESULTS_FIT_DIR / name_folder_save_results
pth_hdegpd_fit_dry_spells = path_folder_save_results / "dry_spell_fit_egpd1_excess_over_1result_fit_parameters.csv"
df_fit_dry = pd.read_csv(pth_hdegpd_fit_dry_spells)
df_fit_dry["city"] = df_fit_dry["data_source"].map(lambda s:s.split()[0])
df_fit_dry["season"] = df_fit_dry["data_source"].map(lambda s:s.split()[-1])
# Load dry-spell durations for Palermo from the JSON exported by notebook 01.
with open(config.EXPORTS_JSON_DIR / 
          f'ecad_data_south_europe_filtered_after_1946_wet_day_thresh_{config.WET_DAY_THRESHOLD}.json') as fh:
    spells = json.load(fh)
list_cities = sorted(spells.keys())
stations = pd.read_csv(config.STATION_METADATA_CSV)
df_info = stations[stations.city.isin(list_cities)]

In [2]:
df_fit_dry_spring = df_fit_dry[df_fit_dry.season =="spring"]
list_d_thresh = [20, 40, 60]
for d_thresh in list_d_thresh:
    list_mean_excess = []
    for city in tqdm(list_cities):
        city = city.split()[0]
        row = df_fit_dry_spring[df_fit_dry_spring.city == city].iloc[0]
        sigma, kappa, xi, f1 = row["sigma"], row["kappa"], row["xi"], row["f_1"]
        approx = make_approx_mean_excess(
            d_thresh=d_thresh,
            sigma=sigma, kappa=kappa, xi=xi, f1=f1,
            target_precision=1e-5)
        list_mean_excess.append(approx)
    df_fit_dry_spring.loc[:, f"mean_excess_{d_thresh}"] = list_mean_excess

# Markov baseline
for d_thresh in list_d_thresh:
    list_mean_excess = []
    for city in tqdm(list_cities):
        full_year_data = spells[city]['dry_spell']['duration_spell']
        full_year_dates = spells[city]['dry_spell']['start_date_spell']
        full_year_months = np.array([str(s)[4:6] for s in full_year_dates])
        month_season = config.LIST_MONTH_SEASONS[1]
        month_set = set(month_season[:3])
        season_mask = np.array([m in month_set for m in full_year_months])
        spring_data = np.array(full_year_data)[season_mask]
        p_geom_dry = 1 / np.mean(spring_data)
        approx = mean_excess_markov_order_1(p_geom_dry, d_thresh)
        list_mean_excess.append(approx)
    df_fit_dry_spring.loc[:, f"mean_excess_{d_thresh}_markov_baseline"] = list_mean_excess

  0%|          | 0/209 [00:00<?, ?it/s]

100%|██████████| 209/209 [00:00<00:00, 252.49it/s]


In [3]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go


list_d_thresh = [20, 40, 60]
df = df_fit_dry_spring.copy()
vmin = 0.1  # mean excess in days — adjust as needed

label_model = "BMCD model"
label_baseline = "Geometric model"
offset, zoom = 0.3, 3.5
needed_cols = ["city", "lat", "lon"]
df_info["city"] = df_info["city"].map(lambda s:s.split()[0])
df = df.merge(df_info,on="city")
for d in list_d_thresh:
    needed_cols += [f"mean_excess_{d}", f"mean_excess_{d}_markov_baseline"]
df = df[needed_cols].dropna(subset=["lat", "lon"]).copy()
all_vals = []
for d in list_d_thresh:
    all_vals.extend(df[f"mean_excess_{d}"].dropna().values)
    all_vals.extend(df[f"mean_excess_{d}_markov_baseline"].dropna().values)
all_vals = np.array(all_vals, dtype=float)
all_vals = all_vals[np.isfinite(all_vals) & (all_vals > 0)]
vmax = all_vals.max()
lat_min, lat_max = df["lat"].min(), df["lat"].max()
lon_min, lon_max = df["lon"].min(), df["lon"].max()
fig = make_subplots(rows=len(list_d_thresh), cols=1,
    specs=[[{"type": "scattermapbox"}] for _ in list_d_thresh],
    subplot_titles=[f"Mean residual duration after {d} dry days" for d in list_d_thresh],
    vertical_spacing=0.02)
for i, d_thresh in enumerate(list_d_thresh, start=1):
    col_model = f"mean_excess_{d_thresh}"
    col_base = f"mean_excess_{d_thresh}_markov_baseline"
    model_vals = df[col_model].astype(float)
    base_vals = df[col_base].astype(float)
    fig.add_trace(
        go.Scattermapbox(lon=df["lon"], lat=df["lat"], mode="markers",
            name=label_baseline if i == 1 else None, showlegend=(i == 1),
            marker=dict(size=16, symbol="circle", color=base_vals,
                        coloraxis="coloraxis", opacity=1),
            text=[f"City: {city}<br>{label_baseline}: {val:.2f} days<br>Threshold: {d_thresh}"
                  for city, val in zip(df["city"], base_vals)],
            hovertemplate="%{text}<extra></extra>"), row=i, col=1)
    fig.add_trace(
        go.Scattermapbox(lon=df["lon"], lat=df["lat"], mode="markers",
            name=label_model if i == 1 else None, showlegend=(i == 1),
            marker=dict(size=8, symbol="circle", color=model_vals,
                        coloraxis="coloraxis", opacity=1),
            text=[f"City: {city}<br>{label_model}: {val:.2f} days<br>Threshold: {d_thresh}"
                  for city, val in zip(df["city"], model_vals)],
            hovertemplate="%{text}<extra></extra>"), row=i, col=1)
center_lat, center_lon = (lat_min + lat_max*1.1) / 2, (lon_min + lon_max) / 2
for i in range(1, len(list_d_thresh) + 1):
    map_name = "mapbox" if i == 1 else f"mapbox{i}"
    fig.update_layout({map_name: dict(style="open-street-map",
        center=dict(lat=center_lat, lon=center_lon), zoom=zoom)})
fig.update_layout(height=1200, width=900, margin=dict(l=20, r=20, t=60, b=20),
    legend=dict(orientation="h", yanchor="bottom", y=0.001, xanchor="center", x=0.81),
    coloraxis=dict(cmin=vmin, cmax=100, colorscale=[[0, "blue"],[0.5, "yellow"],[0.75, "orange"], [1, "red"]],
        colorbar=dict(title="Mean<br>residual<br>duration <br>(days)")))
fig.show()

C:\Users\antoi\AppData\Local\Temp\ipykernel_12196\6920235.py:37: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  go.Scattermapbox(lon=df["lon"], lat=df["lat"], mode="markers",
C:\Users\antoi\AppData\Local\Temp\ipykernel_12196\6920235.py:45: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  go.Scattermapbox(lon=df["lon"], lat=df["lat"], mode="markers",
C:\Users\antoi\AppData\Local\Temp\ipykernel_12196\6920235.py:37: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  go.Scattermapbox(lon=df["lon"], lat=df["lat"], mode="markers",
C:\Users\antoi\AppData\Local\Temp\ipykernel_12196\6920235.py:45: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-map